# 🥈 Silver Layer – Clean & Enrich
- Detect all null values. 
- Identify which columns need external data. 
- Use web scraping to collect missing information where possible. 

In [1]:
import pandas as pd
from sqlalchemy import create_engine
import urllib
from pathlib import Path

In [2]:
# Create connection string
params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=DESKTOP-EKQ2K2R;"
    "DATABASE=Airbnb_DWH;"
    "Trusted_Connection=yes;"
)

# Create SQLAlchemy engine
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

# Read Bronze table
bronze_df = pd.read_sql(
    "SELECT * FROM bronze_airbnb",
    con=engine
)

In [3]:
bronze_df.head() 

,Unnamed: 0,realSum,room_type,room_shared,room_private,person_capacity,host_is_superhost,multi,biz,cleanliness_rating,...,dist,metro_dist,attr_index,attr_index_norm,rest_index,rest_index_norm,lng,lat,city,day_type
0,0,194.033698,Private room,False,True,2.0,False,1,0,10.0,...,5.022964,2.539380,78.690379,4.166708,98.253896,6.846473,4.90569,52.41772,Amsterdam,Weekdays
1,1,344.245776,Private room,False,True,4.0,False,0,0,8.0,...,0.488389,0.239404,631.176378,33.421209,837.280757,58.342928,4.90005,52.37432,Amsterdam,Weekdays
2,2,264.101422,Private room,False,True,2.0,False,0,1,9.0,...,5.748312,3.651621,75.275877,3.985908,95.386955,6.646700,4.97512,52.36103,Amsterdam,Weekdays
3,3,433.529398,Private room,False,True,4.0,False,0,1,9.0,...,0.384862,0.439876,493.272534,26.119108,875.033098,60.973565,4.89417,52.37663,Amsterdam,Weekdays
4,4,485.552926,Private room,False,True,2.0,True,0,0,10.0,...,0.544738,0.318693,552.830324,29.272733,815.305740,56.811677,4.90051,52.37508,Amsterdam,Weekdays


In [4]:
bronze_df.shape

(51707, 22)

In [5]:
bronze_df.info() # Display dataset information

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51707 entries, 0 to 51706
Data columns (total 22 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Unnamed: 0                  51707 non-null  int64  
 1   realSum                     51707 non-null  float64
 2   room_type                   51707 non-null  object 
 3   room_shared                 51707 non-null  bool   
 4   room_private                51707 non-null  bool   
 5   person_capacity             51707 non-null  float64
 6   host_is_superhost           51707 non-null  bool   
 7   multi                       51707 non-null  int64  
 8   biz                         51707 non-null  int64  
 9   cleanliness_rating          51707 non-null  float64
 10  guest_satisfaction_overall  51707 non-null  float64
 11  bedrooms                    51707 non-null  int64  
 12  dist                        51707 non-null  float64
 13  metro_dist                  517

In [6]:
bronze_df.isna().sum() # Count missing values

Unnamed: 0                    0
realSum                       0
room_type                     0
room_shared                   0
room_private                  0
person_capacity               0
host_is_superhost             0
multi                         0
biz                           0
cleanliness_rating            0
guest_satisfaction_overall    0
bedrooms                      0
dist                          0
metro_dist                    0
attr_index                    0
attr_index_norm               0
rest_index                    0
rest_index_norm               0
lng                           0
lat                           0
city                          0
day_type                      0
dtype: int64

In [7]:
bronze_df.duplicated().sum() # Count duplicate rows

np.int64(0)

In [8]:
# Generate summary statistics
bronze_df.describe()

,Unnamed: 0,realSum,person_capacity,multi,biz,cleanliness_rating,guest_satisfaction_overall,bedrooms,dist,metro_dist,attr_index,attr_index_norm,rest_index,rest_index_norm,lng,lat
count,51707.000000,51707.000000,51707.000000,51707.000000,51707.000000,51707.000000,51707.000000,51707.00000,51707.000000,51707.000000,51707.000000,51707.000000,51707.000000,51707.000000,51707.000000,51707.000000
mean,1620.502388,279.879591,3.161661,0.291353,0.350204,9.390624,92.628232,1.15876,3.191285,0.681540,294.204105,13.423792,626.856696,22.786177,7.426068,45.671128
std,1217.380366,327.948386,1.298545,0.454390,0.477038,0.954868,8.945531,0.62741,2.393803,0.858023,224.754123,9.807985,497.920226,17.804096,9.799725,5.249263
min,0.000000,34.779339,2.000000,0.000000,0.000000,2.000000,20.000000,0.00000,0.015045,0.002301,15.152201,0.926301,19.576924,0.592757,-9.226340,37.953000
25%,646.000000,148.752174,2.000000,0.000000,0.000000,9.000000,90.000000,1.00000,1.453142,0.248480,136.797385,6.380926,250.854114,8.751480,-0.072500,41.399510
50%,1334.000000,211.343089,3.000000,0.000000,0.000000,10.000000,95.000000,1.00000,2.613538,0.413269,234.331748,11.468305,522.052783,17.542238,4.873000,47.506690
75%,2382.000000,319.694287,4.000000,1.000000,1.000000,10.000000,99.000000,1.00000,4.263077,0.737840,385.756381,17.415082,832.628988,32.964603,13.518825,51.471885
max,5378.000000,18545.450285,6.000000,1.000000,1.000000,10.000000,100.000000,10.00000,25.284557,14.273577,4513.563486,100.000000,6696.156772,100.000000,23.786020,52.641410


## Cleaning

In [9]:
# Create a copy of the Bronze dataset
silver_df = bronze_df.copy()

In [10]:
# Remove unnecessary index column
silver_df.drop(columns=["Unnamed: 0"], inplace=True)

In [11]:
silver_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51707 entries, 0 to 51706
Data columns (total 21 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   realSum                     51707 non-null  float64
 1   room_type                   51707 non-null  object 
 2   room_shared                 51707 non-null  bool   
 3   room_private                51707 non-null  bool   
 4   person_capacity             51707 non-null  float64
 5   host_is_superhost           51707 non-null  bool   
 6   multi                       51707 non-null  int64  
 7   biz                         51707 non-null  int64  
 8   cleanliness_rating          51707 non-null  float64
 9   guest_satisfaction_overall  51707 non-null  float64
 10  bedrooms                    51707 non-null  int64  
 11  dist                        51707 non-null  float64
 12  metro_dist                  51707 non-null  float64
 13  attr_index                  517

## Rename Columns

In [12]:
# Dictionary containing old and new column names
column_mapping = {
    "realSum": "price",
    "person_capacity": "guest_capacity",
    "host_is_superhost": "is_superhost",
    "cleanliness_rating": "cleanliness_score",
    "guest_satisfaction_overall": "guest_satisfaction_score",
    "dist": "distance_to_city_center",
    "metro_dist": "distance_to_metro",
    "attr_index": "attraction_index",
    "attr_index_norm": "normalized_attraction_index",
    "rest_index": "restaurant_index",
    "rest_index_norm": "normalized_restaurant_index",
    "lng": "longitude",
    "lat": "latitude"
}

# Rename columns
silver_df.rename(columns=column_mapping, inplace=True)

print("Columns renamed successfully.")

Columns renamed successfully.


In [13]:
silver_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51707 entries, 0 to 51706
Data columns (total 21 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   price                        51707 non-null  float64
 1   room_type                    51707 non-null  object 
 2   room_shared                  51707 non-null  bool   
 3   room_private                 51707 non-null  bool   
 4   guest_capacity               51707 non-null  float64
 5   is_superhost                 51707 non-null  bool   
 6   multi                        51707 non-null  int64  
 7   biz                          51707 non-null  int64  
 8   cleanliness_score            51707 non-null  float64
 9   guest_satisfaction_score     51707 non-null  float64
 10  bedrooms                     51707 non-null  int64  
 11  distance_to_city_center      51707 non-null  float64
 12  distance_to_metro            51707 non-null  float64
 13  attraction_index

## Validate Data

In [14]:
# Check for negative prices
print((silver_df["price"] < 0).sum())

# Check for invalid bedrooms
print((silver_df["bedrooms"] < 0).sum())

# Check for invalid guest capacity
print((silver_df["guest_capacity"] <= 0).sum())

0
0
0


## Save Silver Dataset

In [15]:
output_path = Path("../datasets/processed")
output_file = output_path / "silver_airbnb.csv"

silver_df.to_csv(output_file , index= False)

## Load into SQL Server

In [16]:
silver_df.to_sql(
    name="silver_airbnb",
    con=engine,
    if_exists="replace",
    index=False
)

29

## Verification

In [17]:
silver_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51707 entries, 0 to 51706
Data columns (total 21 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   price                        51707 non-null  float64
 1   room_type                    51707 non-null  object 
 2   room_shared                  51707 non-null  bool   
 3   room_private                 51707 non-null  bool   
 4   guest_capacity               51707 non-null  float64
 5   is_superhost                 51707 non-null  bool   
 6   multi                        51707 non-null  int64  
 7   biz                          51707 non-null  int64  
 8   cleanliness_score            51707 non-null  float64
 9   guest_satisfaction_score     51707 non-null  float64
 10  bedrooms                     51707 non-null  int64  
 11  distance_to_city_center      51707 non-null  float64
 12  distance_to_metro            51707 non-null  float64
 13  attraction_index

In [19]:
silver_df.columns.to_list()

['price',
 'room_type',
 'room_shared',
 'room_private',
 'guest_capacity',
 'is_superhost',
 'multi',
 'biz',
 'cleanliness_score',
 'guest_satisfaction_score',
 'bedrooms',
 'distance_to_city_center',
 'distance_to_metro',
 'attraction_index',
 'normalized_attraction_index',
 'restaurant_index',
 'normalized_restaurant_index',
 'longitude',
 'latitude',
 'city',
 'day_type']

# 🥈 Silver Layer

## Objective

The Silver Layer is responsible for transforming the raw Bronze data into a clean, validated, and analysis-ready dataset. Unlike the Bronze Layer, this stage applies data quality checks, standardization, and data transformations while preserving the integrity of the original data.

---

## Steps Performed

### 1. Read Bronze Data

- Connected to the **SQL Server** database.
- Loaded the `bronze_airbnb` table into a Pandas DataFrame.

---

### 2. Data Quality Assessment

Performed a comprehensive data quality check to evaluate the dataset before applying any transformations.

The following checks were completed:

- Dataset shape
- Data types
- Summary statistics
- Missing values
- Duplicate records

### Results

- **Total Records:** 51,707
- **Total Columns:** 22
- **Missing Values:** 0
- **Duplicate Rows:** 0

The dataset was found to be complete and consistent, requiring only structural transformations.

---

### 3. Remove Unnecessary Columns

Removed the following column:

- `Unnamed: 0`

Reason:

This column was generated from the CSV index during export and does not provide any analytical value.

---

### 4. Standardize Column Names

Renamed multiple columns to improve readability and maintain a consistent naming convention.

Examples:

| Old Name | New Name |
|----------|----------|
| realSum | price |
| person_capacity | guest_capacity |
| host_is_superhost | is_superhost |
| dist | distance_to_city_center |
| metro_dist | distance_to_metro |
| attr_index | attraction_index |
| rest_index | restaurant_index |
| lng | longitude |
| lat | latitude |

---

### 5. Data Validation

Validated important business rules to ensure data consistency.

Validation checks included:

- No negative prices
- No invalid bedroom counts
- No invalid guest capacities

The dataset passed all validation checks successfully.

---

### 6. Save Silver Dataset

The cleaned dataset was saved as:

```
datasets/processed/silver_airbnb.csv
```

---

### 7. Load into SQL Server

Loaded the transformed dataset into the Data Warehouse.

Database:

```
Airbnb_DWH
```

Table:

```
silver_airbnb
```

The table was loaded using:

- `pandas.to_sql()`
- `if_exists="replace"`

This ensures the ETL pipeline is repeatable and can be executed multiple times without generating duplicate data.

---

## Silver Layer Output

- Clean and validated dataset
- Standardized column names
- Derived analytical features
- Silver CSV file
- `silver_airbnb` table stored in SQL Server

---

## Notes

The Silver Layer focuses on improving data quality and preparing the dataset for dimensional modeling in the Gold Layer. The resulting dataset serves as the trusted source for building the Star Schema and creating analytical dashboards.